In [103]:
!python3 -V

Python 3.11.7


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [2]:
def read_dataFrame(filePath):
    df = pd.read_parquet(filePath)
    
    #Combining Categorical Feature
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df
    
    

In [3]:
def remove_duration_outliers(df):
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    # Select trips which took between (1-60 miniutes)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    return df
    

In [4]:
#Read data
df_train = read_dataFrame('./data-homework-1/yellow_tripdata_2023-01.parquet')
df_val = read_dataFrame('./data-homework-1/yellow_tripdata_2023-02.parquet')

In [5]:
df_val.shape

(2913955, 19)

In [6]:
# Q1. Number of Columns
df_train.shape[1]

19

In [7]:
#Q2. Compute the standard deviation of column 'duration'
df = df_train
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
std_dev_train = df_train['duration'].std()
std_dev_train

42.594351241920904

In [8]:
# Q3. Dropping outliers
clean_df_train = remove_duration_outliers(df_train)

# What fraction of the records left after you dropped the outliers?
nOfRowsUnclean = df_train.shape[0]
nOfRowsclean = clean_df_train.shape[0]

leftRatio = (nOfRowsclean/nOfRowsUnclean) * 100
leftRatio

98.1220282212598

In [13]:
# Q4. One-hot encoding
categorical = ['PULocationID', 'DOLocationID']
numerical = ['duration']

dv = DictVectorizer()

train_dicts = clean_df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)


df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)
df_val = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)]

val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

# Get the dimensionality of the DataFrame
dimensionality = X_train.shape

# Number of columns
dimensionality[1]


515

In [139]:
#Feature Engineering
#Transforming datetime to Extract Date Components & Cyclical Features for Time Components on Training Dataset 
# clean_df_train['day_of_week'] = clean_df_train['lpep_pickup_datetime'].dt.dayofweek
# clean_df_train['day'] = clean_df_train['lpep_pickup_datetime'].dt.day
# clean_df_train['hour'] = clean_df_train['lpep_pickup_datetime'].dt.hour
# clean_df_train['hour_sin'] = np.sin(2 * np.pi * clean_df_train['hour'] / 24)
# clean_df_train['hour_cos'] = np.cos(2 * np.pi * clean_df_train['hour'] / 24)
# clean_df_train['day_of_week_sin'] = np.sin(2 * np.pi * clean_df_train['day_of_week'] / 7)
# clean_df_train['day_of_week_cos'] = np.cos(2 * np.pi * df_train['day_of_week'] / 7)

#Transforming datetime to Extract Date Components & Cyclical Features for Time Components on Valuation Dataset 
# df_val['day_of_week'] = df_val['lpep_pickup_datetime'].dt.dayofweek
# df_val['day'] = df_val['lpep_pickup_datetime'].dt.day
# df_val['hour'] = df_val['lpep_pickup_datetime'].dt.hour
# df_val['hour_sin'] = np.sin(2 * np.pi * df_val['hour'] / 24)
# df_val['hour_cos'] = np.cos(2 * np.pi * df_val['hour'] / 24)
# df_val['day_of_week_sin'] = np.sin(2 * np.pi * df_val['day_of_week'] / 7)
# df_val['day_of_week_cos'] = np.cos(2 * np.pi * df_val['day_of_week'] / 7)


In [140]:
# #Processing data
# df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
# df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

# categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
# # numerical = ['trip_distance', 'day', 'day_of_week', 'hour', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']
# numerical = ['trip_distance', 'day_of_week', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']


# dv = DictVectorizer()

# train_dicts = df_train[categorical + numerical].to_dict(orient='records')
# X_train = dv.fit_transform(train_dicts)

# val_dicts = df_val[categorical + numerical].to_dict(orient='records')
# X_val = dv.transform(val_dicts)

In [10]:
# Q5. Training a base model (Simple LinearRegression Model)
# Now let's use the feature matrix from the previous step to train a model.
# Train a plain linear regression model with default parameters
# Calculate the RMSE of the model on the training data
target = 'duration'
y_train = clean_df_train[target].values
y_val = clean_df_train[target].values
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)
root_mean_squared_error(y_val, y_pred)


7.649261927686161

In [11]:
# Q6. Evaluating the model on validation dataset (February 2023)

# Now let's apply this model to the validation dataset (February 2023).
# What's the RMSE on validation?
target = 'duration'
y_train = clean_df_train[target].values
y_val = df_val[target].values

# Training the linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

#Prediction & Evaluating the model
y_pred = lr.predict(X_val)
root_mean_squared_error(y_val, y_pred)

7.811817957524739

In [16]:
lr.intercept_

23.197129450882166

In [147]:
# #Feature Importance
# model = RandomForestRegressor()
# model.fit(X_train, y_train)
# feature_importances = model.feature_importances_
# feature_importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': feature_importances})
# feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
# print(feature_importance_df)